# 4.4 FA 算子接口介绍：FA 推理接口逐参解读

本节把 FA 推理算子（`FusedInferAttentionScore`）的接口签名逐参读透。第 3 章 3.2 节给过参数总表，这里从**使用者视角**展开：每个参数怎么填、有什么约束、填错会怎样。

## 1. 接口签名（aclnn 视角）

```cpp
aclnnStatus aclnnFusedInferAttentionScoreGetWorkspaceSize(
    const aclTensor *query,                       // Q，[B, S1, (N1, D) 或 H]
    const aclTensor *key,                         // K
    const aclTensor *value,                       // V
    const aclTensor *pseShift,                    //（可选）位置编码偏移
    const aclTensor *dropMask,                    //（可选）dropout 掩码
    const aclTensor *paddingMask,                 //（可选）padding 掩码
    const aclTensor *attenMask,                   //（可选）注意力掩码
    const aclTensor *actualSeqLengths,            // 每条序列实际长度（Q 侧）
    const aclTensor *actualSeqLengthsKv,          // 每条序列实际长度（KV 侧）
    const aclTensor *deqScale1, const aclTensor *quantScale1, ...   // 量化相关（第 6 章）
    double scaleValue,                            // softmax 缩放系数，通常 1/sqrt(D)
    double preTokens, double nextTokens,          // 掩码边界
    char  inputLayout,                            // 排布：BSH / BNSD / BSND / TND
    int64_t numHeads,                             // Q 头数 N1
    int64_t numKeyValueHeads,                     // KV 头数 N2（GQA）
    ...
    const aclTensor *attentionOut,                // 输出
    uint64_t *workspaceSize, aclOpExecutor **executor);
```

> 参数虽多，但可以分成四组：**数据**（Q/K/V/输出）、**掩码**（三种 mask + 边界）、**形状语义**（头数、实际长度、排布）、**量化**（第 6 章再讲）。

## 2. 数据组：query / key / value / attentionOut

| 参数 | 约束要点 |
|--|--|
| `query` | S1 是 Q 序列长度；推理 Decode 阶段每步 S1=1 |
| `key` / `value` | 两者 shape 必须一致；S2 为 KV 有效长度（含 KV Cache） |
| `attentionOut` | shape 与 query 同构（S1 维替换为 S1） |

S1 与 S2 可以不同（第 3 章 3.3 节的三场景）：Prefill 相等、Decode S1=1、稀疏场景 S1>S2。

## 3. 掩码组：pseShift / dropMask / paddingMask / attenMask + 边界

| 参数 | 语义 | 对应第 2 章概念 |
|--|--|--|
| `pseShift` | 相对位置编码偏置加到分数上（如 ALiBi） | — |
| `dropMask` | 训练用 dropout 掩码（推理一般不传） | 2.5 |
| `paddingMask` | 标记 padding 位置的无效 token | 2.3 |
| `attenMask` | 自定义注意力掩码 | 2.5 因果掩码 |
| `preTokens` / `nextTokens` | 只关注前 preTokens、后 nextTokens 范围 | 带状掩码 |

典型取值：因果掩码 `preTokens=很大值, nextTokens=0`；带状注意力两者都取小值。

## 4. 形状语义组：头数 / 实际长度 / 排布

### numHeads 与 numKeyValueHeads（GQA 入口）

```text
N1 = numHeads（Q 头数），N2 = numKeyValueHeads（KV 头数）
N1 = N2        → MHA（标准多头）
N2 = 1         → MQA（2.9 节）
N2 < N1, N1%N2=0 → GQA，组数 G = N1/N2（2.3 节）
```

### actualSeqLengths / actualSeqLengthsKv

配合变长 batch：BSH/BSND 类排布 padding 到最长 S，真实长度靠这两个数组告诉算子；TND 排布下是叠加式（第 3 章 3.3 节）。

### inputLayout

`BSH / BNSD / BSND / TND`，决定算子内部按哪种 stride 取基本块。

## 5. scaleValue

一般传 `1/sqrt(D)`。第 2 章 2.3 节推导过：点积方差随 D 线性增长，不缩放会导致 Softmax 饱和、梯度消失。为什么做成参数而不写死？——支持 ALiBi / RoPE 变体的自定义缩放。

## 6. 一个典型的 Decode 调用示例

```python
# torch_npu 侧的等价高层接口（内部走到 aclnn FA）
import torch, torch_npu

B, S2, D = 4, 1024, 128          # batch、KV 缓存长度、头维度
N1, N2 = 32, 8                   # 32 个 Q 头共享 8 组 KV（GQA）
q  = torch.randn(B, 1,  N1, D, dtype=torch.float16).npu()      # S1=1：每步 1 个新 token
kv_cache = torch.randn(B, S2, N2, 2, D, dtype=torch.float16).npu()

out = torch_npu.npu_fusion_attention(
    q, kv_cache[..., 0, :], kv_cache[..., 1, :],
    head_num=N1,                          # numHeads
    input_layout="BNSD",
    scale=1.0 / D ** 0.5,                  # scaleValue
    pre_tokens=S2, next_tokens=0,          # 因果掩码
)[0]
```

> 注意接口层的「用户友好」：torch_npu 把 `numKeyValueHeads` 从 K 的 shape 里自动推断（GQA），把排布、缩放、掩码边界显式传参——这些正是 3.5 节 Tiling 第①步解析的东西。

## 小测验

1. `numHeads=32`、`numKeyValueHeads=8`、`inputLayout="BNSD"`，Q 的 shape 应该是几维？KV 呢？
2. 要实现「每个 token 只能看它自己之前 512 个 token」，preTokens / nextTokens 怎么填？
3. Decode 阶段 q 的 S1 是多少？这会怎样影响 Tiling 的核间任务总数？